## Databricks Prompt Evaluation

### Installing Utilities and Libraries

In [ ]:
%pip install --upgrade "mlflow[databricks]>=3.1.0" databricks-sdk==0.77.0 openai

### Restart the Python Environment

In [ ]:
dbutils.library.restartPython()

### Set up a Tacking Experiment

In [ ]:
import mlflow
import uuid

mlflow.set_tracking_uri("databricks")

mlflow.set_experiment(
    "/Shared/ticket-classification-evaluation"
)

# Get current catalog
catalog_name = spark.sql(
    "SELECT current_catalog()"
).collect()[0][0]

schema_name = "default"

suffix = uuid.uuid4().hex[:6]

PROMPT_NAME = (
    f"{catalog_name}.{schema_name}."
    f"ticket_classifier_{suffix}"
)

EVAL_DATASET_NAME = (
    f"{catalog_name}.{schema_name}."
    f"ticket_eval_{suffix}"
)

print("Prompt:", PROMPT_NAME)
print("Dataset:", EVAL_DATASET_NAME)

### Create Prompt Version 1

In [ ]:
prompt_v1 = mlflow.genai.register_prompt(
    name=PROMPT_NAME,

    template="""
Classify the following IT support ticket.

Ticket:
{{ticket}}
""",

    commit_message="v1: Basic ticket classification prompt"
)

print(f"Created Version: {prompt_v1.version}")

### Create Improved Prompt Version 2

In [ ]:
prompt_v2 = mlflow.genai.register_prompt(
    name=PROMPT_NAME,

    template="""
You are an enterprise IT support ticket classifier.

Analyze the following support ticket:

{{ticket}}

Classify the ticket using ONLY one of these categories:

- Network
- Authentication
- Hardware
- Software
- Security

Assign ONLY one of these priority levels:

- Low
- Medium
- High
- Critical

Priority Guidelines:

Critical:
Major security incident or widespread business outage.

High:
A serious issue preventing a user or team from working.

Medium:
An issue affecting productivity but with a workaround available.

Low:
Minor issue or informational request.

Return ONLY the following format:

Category: <category>
Priority: <priority>
""",

    commit_message=(
        "v2: Added classification taxonomy, "
        "priority rules and output constraints"
    )
)

print(f"Created Version: {prompt_v2.version}")

### Create the Evaluation Dataset

In [ ]:
eval_dataset = mlflow.genai.datasets.create_dataset(
    uc_table_name=EVAL_DATASET_NAME
)

In [ ]:
evaluation_examples = [

    {
        "inputs": {
            "ticket":
                "I changed my password this morning and "
                "now I cannot log into my corporate account."
        },
        "expectations": {
            "expected_category": "Authentication",
            "expected_priority": "High"
        }
    },

    {
        "inputs": {
            "ticket":
                "The office Wi-Fi is unavailable for everyone "
                "on the third floor and nobody can access "
                "internal applications."
        },
        "expectations": {
            "expected_category": "Network",
            "expected_priority": "Critical"
        }
    },

    {
        "inputs": {
            "ticket":
                "Microsoft Excel crashes whenever I try to "
                "open one particular spreadsheet. Other "
                "spreadsheets work normally."
        },
        "expectations": {
            "expected_category": "Software",
            "expected_priority": "Medium"
        }
    },

    {
        "inputs": {
            "ticket":
                "I received an email asking me to enter my "
                "company password on an unfamiliar website."
        },
        "expectations": {
            "expected_category": "Security",
            "expected_priority": "High"
        }
    },

    {
        "inputs": {
            "ticket":
                "My second monitor occasionally flickers, "
                "but I can continue working normally."
        },
        "expectations": {
            "expected_category": "Hardware",
            "expected_priority": "Low"
        }
    }
]

In [ ]:
eval_dataset = eval_dataset.merge_records(
    evaluation_examples
)

print(
    f"Added {len(evaluation_examples)} "
    "evaluation records."
)

### Create a LLM Model Client

In [ ]:
from databricks.sdk import WorkspaceClient
import openai

# create the LLM client
llm_client = WorkspaceClient().serving_endpoints.get_open_ai_client()

# Define the model name
model_name = "databricks-claude-sonnet-4-5"

### Create the Classification Function

In [ ]:
def create_classifier(prompt_name, version):

    @mlflow.trace
    def classify_ticket(ticket: str):

        prompt = mlflow.genai.load_prompt(
            name_or_uri=(
                f"prompts:/{prompt_name}/{version}"
            )
        )

        formatted_prompt = prompt.format(
            ticket=ticket
        )

        response = llm_client.chat.completions.create(
            model=model_name,
            messages=[
                {
                    "role": "user",
                    "content": formatted_prompt
                }
            ],
            temperature=0.1
        )

        return response.choices[0].message.content

    return classify_ticket

### Create a Classification Judge

In [ ]:
from mlflow.genai import make_judge

classification_judge = make_judge(

    name="classification_accuracy",

    instructions="""
Evaluate whether the generated IT support ticket classification
matches the expected classification.

Generated classification:
{{ outputs }}

Expected classification:
{{ expectations }}

The expectations contain:
- expected_category: the correct ticket category
- expected_priority: the correct ticket priority

Return true ONLY if BOTH:
1. The generated category matches expected_category.
2. The generated priority matches expected_priority.

Otherwise return false.
""",

    feedback_value_type=bool,

    model="databricks:/databricks-claude-sonnet-4-5"
)

### Create an Output-Format Judge

In [ ]:
format_judge = make_judge(

    name="output_format_compliance",

    instructions="""
Evaluate whether the generated output follows exactly
this structure:

Category: <category>
Priority: <priority>

The category must be one of:
Network, Authentication, Hardware, Software, Security

The priority must be one of:
Low, Medium, High, Critical

Generated output:

{{ outputs }}

Return true if the output follows these requirements.
Otherwise return false.
""",

    feedback_value_type=bool,

    model = "databricks:/databricks-claude-sonnet-4-5"
)

### Evaluate Both the Prompts

In [ ]:
scorers = [
    classification_judge,
    format_judge
]

results = {}

for version in [1, 2]:

    print(f"\nEvaluating Version {version}...")

    with mlflow.start_run(
        run_name=f"ticket_classifier_v{version}"
    ):

        mlflow.log_param(
            "prompt_version",
            version
        )

        eval_results = mlflow.genai.evaluate(

            predict_fn=create_classifier(
                PROMPT_NAME,
                version
            ),

            data=eval_dataset,

            scorers=scorers
        )

        results[f"v{version}"] = eval_results

### Compare Metrics

In [ ]:
ACCURACY_METRIC = "classification_accuracy/mean"
FORMAT_METRIC = "output_format_compliance/mean"


def get_metric(result, metric):

    if metric not in result.metrics:
        raise KeyError(
            f"{metric} not found. "
            f"Available metrics: {result.metrics.keys()}"
        )

    return result.metrics[metric]

In [ ]:
print("===== PROMPT COMPARISON =====")

for version, result in results.items():

    accuracy = get_metric(
        result,
        ACCURACY_METRIC
    )

    format_score = get_metric(
        result,
        FORMAT_METRIC
    )

    print(f"\n{version}")

    print(
        f"Classification Accuracy: {accuracy:.2f}"
    )

    print(
        f"Format Compliance: {format_score:.2f}"
    )